In [381]:
%reload_ext autoreload
%autoreload 2

from src.generate_surface_code import SurfaceCode
from src.TN_decoder import decoder
from src.parse_syndrome import *
import stim
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm
import pymatching

In [382]:
distance = 3
noise_model = "depolarize"
noise = 0.0667
chi = 8
nshots = 1000

In [383]:
def count_logical_errors(code, num_shots: int) -> int:

    circuit = code.circuit
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(shots = num_shots, separate_observables=True)
    predictions = []

    for event in tqdm(detection_events):
        predictions.append(decoder(code, event, chi))


    fails = sum([1 if not predictions[j] == observable_flips[j] else 0 for j in range(num_shots)])
    return fails

In [384]:
def count_MWPM_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
    # Sample the circuit.
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    # Configure a decoder using the circuit.
    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    # Run the decoder.
    predictions = matcher.decode_batch(detection_events)

    # Count the mistakes.
    num_errors = 0
    for shot in range(num_shots):
        actual_for_shot = observable_flips[shot]
        predicted_for_shot = predictions[shot]
        if not np.array_equal(actual_for_shot, predicted_for_shot):
            num_errors += 1
    return num_errors

In [385]:
code = SurfaceCode(distance, noise_model, noise)

In [393]:
code.circuit

stim.Circuit('''
    MPP X1*X0*X3 X2*X1*X4 X6*X5*X8*X3 X7*X6*X9*X4 X11*X10*X8 X12*X11*X9 Z3*Z5*Z0 Z4*Z3*Z6*Z1 Z4*Z7*Z2 Z8*Z10*Z5 Z9*Z8*Z11*Z6 Z9*Z12*Z7 Z0*Z1*Z2
    PAULI_CHANNEL_1(0, 0, 0.0667) 0 1 2 3 4 5 6 7 8 9 10 11 12
    MPP X1*X0*X3 X2*X1*X4 X6*X5*X8*X3 X7*X6*X9*X4 X11*X10*X8 X12*X11*X9 Z3*Z5*Z0 Z4*Z3*Z6*Z1 Z4*Z7*Z2 Z8*Z10*Z5 Z9*Z8*Z11*Z6 Z9*Z12*Z7 Z0*Z1*Z2
    DETECTOR(1, 0, 0) rec[-26] rec[-13]
    DETECTOR(3, 0, 0) rec[-25] rec[-12]
    DETECTOR(1, 2, 0) rec[-24] rec[-11]
    DETECTOR(3, 2, 0) rec[-23] rec[-10]
    DETECTOR(1, 4, 0) rec[-22] rec[-9]
    DETECTOR(3, 4, 0) rec[-21] rec[-8]
    DETECTOR(0, 1, 0) rec[-20] rec[-7]
    DETECTOR(2, 1, 0) rec[-19] rec[-6]
    DETECTOR(4, 1, 0) rec[-18] rec[-5]
    DETECTOR(0, 3, 0) rec[-17] rec[-4]
    DETECTOR(2, 3, 0) rec[-16] rec[-3]
    DETECTOR(4, 3, 0) rec[-15] rec[-2]
    OBSERVABLE_INCLUDE(0) rec[-14] rec[-1]
''')

In [386]:
m_errors = count_MWPM_logical_errors(code.circuit, nshots)
print(f"There were {m_errors} wrong predictions out of {nshots} shots")

There were 0 wrong predictions out of 1000 shots


In [387]:
num_errors = count_logical_errors(code, nshots)
print(f"There were {num_errors} wrong predictions out of {nshots} shots")

100%|██████████| 1000/1000 [00:06<00:00, 150.21it/s]

There were 172 wrong predictions out of 1000 shots


In [388]:
event = [False for _ in range(12)]
event[3] = True
event[10] = True

In [389]:
code = SurfaceCode(3,noise_model,noise)

In [390]:
coords = get_active_detector_coordinates(event, code.dem)

In [391]:
error_chain = get_error_chain(coords)

In [392]:
error_chain

[(0, 2, 1), (2, 0, 0), (2, 2, 2)]